# AI Customer Support Agent – Agentic System with LangChain & MCP

## Project Description

This project implements an autonomous customer support agent built on LangChain and OpenAI. The agent handles customer inquiries independently: it decides which information is needed, which tools to call, which model is appropriate for the request – and when escalation is necessary.

Tool integration is handled by a custom MCP server (`mcp_server.py`) running as a separate microservice. Tools are loaded dynamically at runtime – no static configuration in the agent.

---

## Setup & Quickstart

### Requirements

```bash
pip install langchain langchain-core langchain-openai langchain-mcp-adapters
pip install fastmcp python-dotenv openpyxl
```

Optional for GUI:

```bash
pip install ipywidgets voila
```

### Project Structure

```
support_agent/
├── support_agent.ipynb
├── mcp_server.py
├── .env                  ← OPENAI_API_KEY=sk-...
└── data/
    ├── customers.xlsx
    └── products.xlsx
```

### Running

**Step 1 – Start the MCP server** (Terminal 1):

```bash
python mcp_server.py
```

The server runs at `http://127.0.0.1:8000`.

**Step 2a – Launch as a Voilà app** (Terminal 2):

```bash
voila support_agent.ipynb --TagRemovePreprocessor.remove_cell_tags=hide
```

Opens at `http://localhost:8866`.

**Step 2b – Run notebook directly:**
Execute all cells in order (`Run All`).

---

## Agent Architecture

The agent consists of three core components orchestrated in `run_agent()`:

- **Input Parser** – Extracts `customer_id` and `product_id` from the conversation history. Already known IDs are retained across multiple turns.
- **Model Routing** – `choose_model()` evaluates request complexity using the LLM and selects between a fast and a powerful model.
- **Tool Loop** – A `while True` loop calls tools in any order until the agent has no further tool calls and delivers a final response.

### Imports

In [1]:
import pandas as pd
import os
import json

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI

### Load Data

In [2]:
customers = pd.read_excel("data/customers.xlsx")
products = pd.read_excel("data/products.xlsx")

In [3]:
if not os.path.exists("escalations.csv"):
    pd.DataFrame(columns=[
        "timestamp", "data", "reason", "tools_used", "model_used"
    ]).to_csv("escalations.csv", index=False)

## Tool Integration via MCP

Tools are not defined in the notebook but served by `mcp_server.py` as a separate microservice. The agent connects at runtime via `client.get_tools()` and loads the available tools dynamically.

The loaded tools are bound to the LLM via `bind_tools(tools)`. The model decides in the tool loop which tool to call and when.

**Available Tools:**
- `get_customer()` – Retrieve customer data
- `get_product()` – Retrieve product data
- `log_escalation()` – Escalation with structured logging
- `update_customer()` – Update customer fields

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MultiServerMCPClient establishes the connection to the MCP server.
# The agent has no prior knowledge of the tools – it queries the server at runtime.
client = MultiServerMCPClient({
    "support": {
        "url": "http://127.0.0.1:8000/mcp",  # address of the running MCP server
        "transport": "streamable_http"         # HTTP-based transport protocol
    }
})

# get_tools() fetches the tool list dynamically from the server.
# The agent discovers available tools itself – no static definition in the notebook.
tools = await client.get_tools()

## Models

In [5]:
# OpenAI API key is loaded from .env
from dotenv import load_dotenv
load_dotenv()

# Lightweight model – used for the parser (Step 1) and simple requests
simple_llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17")
parser_llm = simple_llm

# Powerful model – used for complex requests, escalations, and tool-heavy interactions
strong_llm = ChatOpenAI(model="gpt-5.4-2026-03-05")

## Dynamic Model Routing

`choose_model()` sends the current conversation history to `simple_llm` and asks it to respond with a single word (`simple` or `complex`) to decide which model is appropriate for the request.

This approach is more robust than keyword-based rules: the agent correctly classifies unusual phrasing like *"The device is making strange noises"* as complex without any explicit rules.

- `gpt-5.4-mini` – for simple information lookups
- `gpt-5.4` – for complex requests, escalations, and angry customers

In [6]:
async def choose_model(user_input, conversation_history):
    """
    LLM-based model routing:
    simple_llm evaluates whether the request is complex.
    """

    # The full conversation history is passed so the agent
    # considers the context of the entire conversation.
    decision_prompt = f"""
    Analyze this support request and decide which model complexity is needed.

    Conversation so far:
    {chr(10).join([f"{type(m).__name__}: {m.content}" for m in conversation_history if hasattr(m, 'content')])}

    New user message: {user_input}

    Reply with ONLY one word:
    - "simple" → general info lookup, no damage, no escalation needed
    - "complex" → physical damage, escalation, unsolvable issue, angry customer
    """

    # simple_llm makes the decision – resource-efficient since only one word is expected
    decision = await simple_llm.ainvoke([HumanMessage(content=decision_prompt)])
    decision_text = decision.content.strip().lower()

    if "complex" in decision_text:
        # Powerful model with tool access for complex cases
        return (
            strong_llm.bind_tools(tools),
            "[Model Routing] → Agent decided: Complex → gpt-5.4-2026-03-05 (strong)"
        )
    else:
        # Lightweight model with tool access for simple requests
        return (
            simple_llm.bind_tools(tools),
            "[Model Routing] → Agent decided: Simple → gpt-5.4-mini-2026-03-17 (lightweight)"
        )

## Agent Logic

**Missing information:** If `customer_id` or `product_id` is missing, the agent asks a targeted follow-up question. The `conversation_history` is carried across all turns – already known IDs are not lost.

**Autonomous decision-making:** The agent chooses based on the situation:
- Data available → generate response
- IDs missing → ask follow-up
- Problem unsolvable → call `log_escalation()`

**Escalation logging:** On escalation, `log_escalation()` writes a structured entry to `escalations.csv` with timestamp, summary, reason, tools used, and model used.

In [7]:
SYSTEM_PROMPT = """
You are a support agent.

You can:
- ask the user for missing information
- use tools when needed
- decide which tool to use
- escalate if necessary

Rules:
- If customer_id is missing → ask for it
- If product_id is missing → ask for it
- Use tools to retrieve customer and product data
- If product information is available → answer based on it
- If problem cannot be solved → use log_escalation tool

When escalating, always fill in ALL parameters:
- data: full summary of the customer request
- reason: why the issue cannot be resolved
- tools_used: which tools were called (e.g. get_customer, get_product)
- model_used: gpt-5.4-mini-2026-03-17 for parsing, gpt-5.4-2026-03-05 for support

Always think step by step.
"""

## Execution

### Logging

```
run_agent() is called
       │
       ├── make_logger(on_think, on_agent, output_widget)
       │        │
       │        └── returns log_think() and log_agent()
       │
       ├── parse_input()  →  log_think("[Parsed Info] ...")
       │
       ├── choose_model() →  log_think("[Model Routing] ...")
       │
       ├── run_tool_loop()
       │        ├── log_think("🔧 Tool: get_customer ...")
       │        ├── log_think("   ↳ customer_id: 2001 ...")
       │        ├── log_think("🔧 Tool: get_product ...")
       │        └── log_think("   ↳ product_id: A200 ...")
       │
       └── log_agent("I have escalated the case...")
 ```

In [8]:
# -----------------------------------------------
# Logging
#
# make_logger() encapsulates the output logic and returns two functions
# that route output to the GUI, a Jupyter widget, or the terminal.

def make_logger(on_think, on_agent, output_widget):

    def log_think(msg):
        # for internal reasoning steps (parsed info, model routing, tool calls)
        if on_think:
            on_think(msg)           # callback for the GUI (add_system_message)
        elif output_widget:
            with output_widget:     # Jupyter output widget
                print(msg)
        else:
            print(msg)              # direct print to terminal

    def log_agent(msg):
        # for the agent's final response
        if on_agent:
            on_agent(msg)           # callback for the GUI (add_agent_message)
        elif output_widget:
            with output_widget:     # Jupyter output widget
                print(msg)
        else:
            print(msg)              # direct print to terminal

    return log_think, log_agent

### Parse Input

- The user message is parsed before each agent call.
- The full conversation history is passed so that already known IDs (`customer_id`, `product_id`) are not lost between turns.

In [9]:
# -----------------------------------------------
# Step 1: Parse Input
# Extracts customer_id and product_id from the conversation history.

async def parse_input(user_input, conversation_history, log_think):

    parsing_prompt = f"""
    You are extracting structured information from a conversation.

    Conversation so far:
    {chr(10).join([f"{type(m).__name__}: {m.content}" for m in conversation_history if hasattr(m, 'content')])}

    New user message:
    {user_input}

    Extract:
    - customer_id (if already known from conversation, use that)
    - product_id (if already known from conversation, use that)

    Return JSON with customer_id and product_id.
    """

    # parser_llm extracts the IDs – no tool binding needed
    parsed_info = await parser_llm.ainvoke([HumanMessage(content=parsing_prompt)])
    parsed_info = parsed_info.content

    try:
        # parse JSON and format for log output
        parsed_dict = json.loads(parsed_info)
        parsed_str = " | ".join(f"{k}: {v}" for k, v in parsed_dict.items() if v)
        log_think(f"[Parsed Info] {parsed_str}")
    except:
        # if no valid JSON is returned – output raw text
        log_think(f"[Parsed Info] {parsed_info}")

    # append parsed info + user input to history
    # so the support agent knows both in the next step
    conversation_history.append(
        HumanMessage(content=f"Parsed Info:\n{parsed_info}\n\nUser Input:\n{user_input}")
    )

    return parsed_info, conversation_history

### Tool Loop

The agent decides on each iteration:
- no tool_calls → response is ready → loop exits
- tool_calls present → execute tools → append result to history → next iteration

This allows any number of tools to be called in any order.

In [10]:
# -----------------------------------------------
# Step 2: Tool Loop
# The agent decides whether to respond, ask a follow-up, or escalate.
# Tools are called on demand – no fixed order.

async def run_tool_loop(active_llm, conversation_history, log_think):

    while True:
        # agent receives the full history and decides what happens next
        response = await active_llm.ainvoke(conversation_history)
        conversation_history.append(response)

        # no tool calls → agent has a final response → exit loop
        if not response.tool_calls:
            break

        # agent returned one or more tool calls
        for tc in response.tool_calls:
            log_think(f"🔧 Tool: {tc['name']} | Args: {tc['args']}")

            # tool_map allows fast lookup by tool name
            tool_map = {t.name: t for t in tools}

            # invoke tool asynchronously (MCP tools are async)
            result = await tool_map[tc["name"]].ainvoke(tc["args"])

            # MCP returns structured objects – extract text content only
            try:
                result_text = next(
                    item["text"] for item in result if item.get("type") == "text"
                )
                parsed_result = eval(result_text)  # string → Python dict
                clean_str = " | ".join(f"{k}: {v}" for k, v in parsed_result[0].items())
                log_think(f"   ↳ {clean_str}")
            except:
                log_think(f"   ↳ {result}")  # fallback: raw output

            # append tool result to history so the agent
            # knows the tool's response on the next iteration
            conversation_history.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    # last response = agent's final answer (no more tool calls)
    return conversation_history, response

### Orchestration

`run_agent()` coordinates all steps and is called once per user message. The updated `conversation_history` is returned and carried forward to the next call.

In [11]:
# -----------------------------------------------
# run_agent – orchestrates all steps

async def run_agent(user_input, conversation_history,
                    output_widget=None, on_think=None, on_agent=None):

    # initialize logger – routes output to GUI or terminal depending on context
    log_think, log_agent = make_logger(on_think, on_agent, output_widget)

    # Step 1: extract customer_id and product_id from message + history
    _, conversation_history = await parse_input(user_input, conversation_history, log_think)

    # Step 2: LLM decides whether to use simple_llm or strong_llm
    active_llm, model_info = await choose_model(user_input, conversation_history)
    log_think(model_info)

    # Step 3: agent runs in tool loop until no further tool calls are needed
    conversation_history, response = await run_tool_loop(active_llm, conversation_history, log_think)

    # output the agent's final response (text, no more tool calls)
    log_agent(response.content)
    return conversation_history  # return history for the next turn

## Voilà GUI

The chat interface is built with `ipywidgets` and served as a standalone web app via Voilà. Internal reasoning steps (tool calls, model routing, parsed IDs) are shown as subtle grey text – visible in the notebook, hidden in the app view.

In [12]:
# -----------------------------------------------
# Imports & Widgets
# Widgets are defined here – layout and event handler follow in the next cells.

import ipywidgets as widgets
from IPython.display import display
import asyncio
import html
import re

customer_input = widgets.Text(
    placeholder="e.g. 2001",
    description="Customer ID:",
    layout=widgets.Layout(width="350px")
)
product_input = widgets.Text(
    placeholder="e.g. A200",
    description="Product ID:",
    layout=widgets.Layout(width="350px")
)
message_input = widgets.Textarea(
    placeholder="Your message...",
    description="Message:",
    layout=widgets.Layout(width="100%", height="80px")
)
send_button = widgets.Button(
    description="Send ➤",
    button_style="primary",
    layout=widgets.Layout(width="100%")
)
chat_html = widgets.HTML(
    layout=widgets.Layout(width="100%")
)

In [13]:
# -----------------------------------------------
# Chat Rendering & Message Functions

# chat_messages stores all messages as HTML strings in a list.
# render_chat() is called on every new message and rewrites chat_html.

chat_messages = []

def md_to_html(text):
    text = html.escape(text)
    text = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', text)
    text = re.sub(r'\*(.*?)\*', r'<em>\1</em>', text)
    text = text.replace("\n", "<br>")
    return text

def render_chat():
    # join all stored messages and write to chat_html.
    # flex-direction:column-reverse ensures new messages appear
    # at the bottom without JavaScript scroll logic.
    content = "".join(chat_messages)
    chat_html.value = f"""
    <div style='min-height:400px; max-height:70vh; overflow-y:auto; overflow-x:hidden;
         padding:10px; width:100%; box-sizing:border-box;
         background:#f9f9f9; border:1px solid #ddd; border-radius:8px;
         font-family:sans-serif; font-size:13px;
         display:flex; flex-direction:column-reverse;'>
        <div>{content}</div>
    </div>
    """

def add_user_message(msg):
    # green bubble right – user message
    chat_messages.append(f"""
        <div style='text-align:right; margin:6px 0;'>
            <span style='background:#28a745; color:white; padding:8px 12px;
                         border-radius:16px 16px 2px 16px; display:inline-block;
                         max-width:75%; text-align:left;'>
                {md_to_html(msg)}
            </span>
        </div>
    """)
    render_chat()

def add_agent_message(msg):
    # blue bubble left – agent response
    chat_messages.append(f"""
        <div style='text-align:left; margin:6px 0;'>
            <span style='background:#0d6efd; color:white; padding:8px 12px;
                         border-radius:16px 16px 16px 2px; display:inline-block;
                         max-width:75%; text-align:left;'>
                {md_to_html(msg)}
            </span>
        </div>
    """)
    render_chat()

def add_system_message(msg):
    # grey centered text – internal reasoning steps (parsed info, tool calls, model routing)
    chat_messages.append(f"""
        <div style='text-align:center; margin:4px 0;'>
            <span style='color:#999; font-size:11px; font-style:italic;'>
                {html.escape(msg).replace(chr(10), "<br>")}
            </span>
        </div>
    """)
    render_chat()

In [14]:
# -----------------------------------------------
# Event Handler & Display

# gui_history stores the full conversation.
# Initialized once with the system prompt and carried across all turns.
gui_history = [SystemMessage(content=SYSTEM_PROMPT)]

async def on_send_async(b):
    global gui_history
    send_button.disabled = True  # disable button while agent is running – prevents double clicks

    parts = []
    if customer_input.value.strip():
        parts.append(f"customer_id: {customer_input.value.strip()}")
    if product_input.value.strip():
        parts.append(f"product_id: {product_input.value.strip()}")
    if message_input.value.strip():
        parts.append(message_input.value.strip())

    # nothing entered → re-enable button and abort
    if not parts:
        send_button.disabled = False
        return

    user_input = "\n".join(parts)
    message_input.value = ""        # clear message field after sending
    add_user_message(user_input)    # show user message immediately in chat

    # call agent – GUI callbacks are passed as on_think and on_agent
    # so log_think() → add_system_message() and log_agent() → add_agent_message()
    gui_history = await run_agent(
        user_input, gui_history,
        on_think=add_system_message,
        on_agent=add_agent_message
    )
    send_button.disabled = False    # re-enable button

def on_send(b):
    # Jupyter already has a running event loop – asyncio.ensure_future()
    # schedules the async function without blocking
    asyncio.ensure_future(on_send_async(b))

send_button.on_click(on_send)       # bind button to event handler

# display all widgets in a vertical box
display(widgets.VBox(
    [customer_input, product_input, chat_html, message_input, send_button],
    layout=widgets.Layout(width="100%", overflow="hidden")
))